# Satellite Wind Safe Multi-Horizon 24h v10 Training

GPU 서버에서 위에서 아래로 실행한다.

- `/home/j-k14s305/s305-work` 아래 업로드한 기존 24h zip 압축 해제
- 데이터 번들은 그대로 쓰고, 학습 코드/모델 구조와 solar/weather/cloud consistency weight를 v10으로 변경
- 1h~24h multi-horizon bundle metadata 검증
- `satellite_wind_safe_multihorizon_24h_v10_solar_weather_cloud_weighted` 학습
- horizon-balanced loss + solar/weather/cloud consistency weight + short/long head 검증
- 24개 horizon/region/worst case 결과 출력


In [8]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import gc
import json
import random
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

WORK = Path("/home/j-k14s305/s305-work")
BUNDLE_NAME = "satellite_wind_safe_multihorizon_24h_regions_2025_20260508_095509"
ZIP = WORK / f"{BUNDLE_NAME}.zip"
OUT = WORK / "runs" / "satellite_wind_safe_multihorizon_24h_v10_solar_weather_cloud_weighted"
OUT.mkdir(parents=True, exist_ok=True)

SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BATCH_SIZE = 4096
EPOCHS = 55
PATIENCE = 10
MIN_DELTA = 5e-5
LR = 2e-4
NUM_WORKERS = 8
FORCE_RETRAIN = False

BASE_NUM_COLS = [
    "cap_scaled",
    "solar_elev_scaled",
    "is_daylight",
    "hour_scaled",
    "doy_scaled",
    "month_scaled",
    "hour_of_day_sin",
    "hour_of_day_cos",
    "day_of_year_sin",
    "day_of_year_cos",
]

WIND_SAFE_COLS = [
    "wind_u_scaled",
    "wind_v_scaled",
    "wind_speed_scaled",
    "wind_dir_sin",
    "wind_dir_cos",
    "asos_ta_scaled",
    "asos_hm_scaled",
    "asos_rn_log1p",
]

HORIZON_CONTEXT_COLS = [
    "horizon_hours_scaled",
    "horizon_hours_sin",
    "horizon_hours_cos",
    "anchor_hour_sin",
    "anchor_hour_cos",
    "anchor_day_of_year_sin",
    "anchor_day_of_year_cos",
]

EXPERIMENTS = [
    ("satellite_wind_safe_multihorizon_24h_v10_solar_weather_cloud_weighted", BASE_NUM_COLS + WIND_SAFE_COLS + HORIZON_CONTEXT_COLS),
]


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = True


def resolve_data_dir() -> Path:
    if not ZIP.exists():
        raise FileNotFoundError(ZIP)

    print("zip:", ZIP.exists(), ZIP)
    with zipfile.ZipFile(ZIP, "r") as zf:
        zf.extractall(WORK)

    markers = sorted(WORK.glob("**/metadata/samples_modeling_all_wind_safe_multihorizon_24h_*.parquet"))
    candidates = [m.parents[1] for m in markers if BUNDLE_NAME in str(m.parents[1])]
    if not candidates:
        raise FileNotFoundError("24h multi-horizon bundle metadata를 찾지 못했다. zip 업로드/압축해제 상태를 확인해야 한다.")

    data_dir = candidates[0]
    print("DATA:", data_dir)
    print("metadata parquet:", len(list((data_dir / "metadata").glob("*.parquet"))))
    print("image shards:", len(list((data_dir / "images").glob("*.npz"))))

    summary_path = data_dir / "metadata" / "multihorizon_24h_bundle_summary.json"
    if summary_path.exists():
        summary = json.loads(summary_path.read_text(encoding="utf-8"))
        print("bundle generated_at:", summary.get("generated_at"))
        print("horizons:", summary.get("horizons"))
        print("strong_filter rows:", summary.get("variant_counts", {}).get("strong_filter"))
    return data_dir


seed_everything(SEED)
DATA = resolve_data_dir()

print("device:", DEVICE)
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


zip: True /home/j-k14s305/s305-work/satellite_wind_safe_multihorizon_24h_regions_2025_20260508_095509.zip
DATA: /home/j-k14s305/s305-work/satellite_wind_safe_multihorizon_24h_regions_2025_20260508_095509
metadata parquet: 10
image shards: 12
bundle generated_at: 2026-05-08T09:55:23
horizons: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24]
strong_filter rows: {'all': 283651, 'train': 234694, 'val': 48957, 'horizons': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24]}
device: cuda
gpu: NVIDIA H200 NVL


In [9]:
def add_scaled_cols(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    out["target_timestamp_kst"] = pd.to_datetime(out["target_timestamp_kst"])
    out["anchor_timestamp_kst"] = pd.to_datetime(out["anchor_timestamp_kst"])
    out["cap_scaled"] = (out["estimated_capacity_kw"] / 300000.0).astype("float32")
    out["solar_elev_scaled"] = (out["solar_elevation"] / 90.0).astype("float32")
    out["hour_scaled"] = (out["hour"] / 23.0).astype("float32")
    out["doy_scaled"] = (out["day_of_year"] / 366.0).astype("float32")
    out["month_scaled"] = (out["month"] / 12.0).astype("float32")
    out["is_daylight"] = out["is_daylight"].astype("float32")

    horizon = out["horizon_hours"].astype(float)
    horizon_rad = 2 * np.pi * horizon / 24.0
    out["horizon_hours_scaled"] = ((horizon - 1.0) / 23.0).astype("float32")
    out["horizon_hours_sin"] = np.sin(horizon_rad).astype("float32")
    out["horizon_hours_cos"] = np.cos(horizon_rad).astype("float32")

    anchor_hour = out["anchor_timestamp_kst"].dt.hour.astype(float)
    anchor_doy = out["anchor_timestamp_kst"].dt.dayofyear.astype(float)
    anchor_hour_rad = 2 * np.pi * anchor_hour / 24.0
    anchor_doy_rad = 2 * np.pi * anchor_doy / 366.0
    out["anchor_hour_sin"] = np.sin(anchor_hour_rad).astype("float32")
    out["anchor_hour_cos"] = np.cos(anchor_hour_rad).astype("float32")
    out["anchor_day_of_year_sin"] = np.sin(anchor_doy_rad).astype("float32")
    out["anchor_day_of_year_cos"] = np.cos(anchor_doy_rad).astype("float32")
    return out


def load_frames():
    train = pd.read_parquet(DATA / "metadata" / "samples_daylight_strong_filter_train.parquet")
    clean_val = pd.read_parquet(DATA / "metadata" / "samples_daylight_strong_filter_val.parquet")
    real_val = pd.read_parquet(DATA / "metadata" / "samples_daylight_no_filter_val.parquet")

    region_map = {name: idx for idx, name in enumerate(sorted(train["region"].unique()))}
    horizon_map = {h: h - 1 for h in range(1, 25)}

    frames = []
    for frame in [train, clean_val, real_val]:
        frame = add_scaled_cols(frame)
        frame["image_file_key"] = frame["image_file"].map(lambda value: Path(value).name)
        frame["region_id"] = frame["region"].map(region_map).astype("int64")
        frame["horizon_id"] = frame["horizon_hours"].astype(int).map(horizon_map).astype("int64")
        frames.append(frame)

    train, clean_val, real_val = frames
    horizon_counts = train["horizon_hours"].astype(int).value_counts().sort_index()
    mean_horizon_count = float(horizon_counts.mean())
    horizon_balance = {int(h): mean_horizon_count / float(count) for h, count in horizon_counts.items()}
    for frame in [train, clean_val, real_val]:
        frame["horizon_balance_weight"] = frame["horizon_hours"].astype(int).map(horizon_balance).fillna(1.0).astype("float32")
        base_weight = frame.get("sample_weight_strong", pd.Series(1.0, index=frame.index)).astype("float32")
        frame["base_train_weight"] = (base_weight * frame["horizon_balance_weight"]).clip(0.05, 5.0).astype("float32")
        frame["solar_weather_cloud_weight"] = np.float32(1.0)
        frame["train_weight"] = frame["base_train_weight"]

    fair_keys = (
        real_val.groupby(["region", "target_timestamp_kst"])["horizon_hours"]
        .nunique()
        .reset_index()
        .query("horizon_hours == 24")[["region", "target_timestamp_kst"]]
    )
    real_fair = real_val.merge(fair_keys, on=["region", "target_timestamp_kst"], how="inner")
    return train, clean_val, real_val, real_fair, region_map, horizon_map


def validate_columns(frame: pd.DataFrame, cols: list[str], name: str) -> None:
    missing = [col for col in cols if col not in frame.columns]
    if missing:
        raise KeyError(f"{name} missing columns: {missing}")
    nulls = frame[cols].isna().sum()
    bad = nulls[nulls > 0]
    if len(bad):
        raise ValueError(f"{name} has null model columns:\n{bad}")


train_df, clean_val_df, real_val_df, real_fair_df, region_map, horizon_map = load_frames()
all_model_cols = sorted({col for _, cols in EXPERIMENTS for col in cols})

for frame_name, frame in {
    "train": train_df,
    "clean_val": clean_val_df,
    "real_val": real_val_df,
    "real_fair": real_fair_df,
}.items():
    validate_columns(frame, all_model_cols, frame_name)

print("train:", len(train_df))
print("clean val strong:", len(clean_val_df))
print("real val no_filter:", len(real_val_df))
print("real fair val:", len(real_fair_df), "/", len(real_val_df))
print("regions:", region_map)
print("horizons:", horizon_map)

print("\nfeature ranges:")
print(train_df[all_model_cols].describe().T[["min", "mean", "max"]].to_string())

shards = {}
for path in sorted((DATA / "images").glob("*.npz")):
    shards[path.name] = np.load(path)["images"]

ram_gb = sum(arr.nbytes for arr in shards.values()) / 1024**3
print("\npreloaded image shards:", len(shards), "RAM GB:", round(ram_gb, 2))



def build_cloud_stats(frame: pd.DataFrame) -> pd.DataFrame:
    keys = frame[["image_file_key", "image_row"]].drop_duplicates().copy()
    rows = []
    for image_file, part in keys.groupby("image_file_key", sort=False):
        row_ids = part["image_row"].astype("int64").to_numpy()
        ca_chunks = []
        cf_chunks = []
        cld_chunks = []
        for start in range(0, len(row_ids), 512):
            batch_rows = row_ids[start:start + 512]
            image = shards[image_file][batch_rows].astype("float32", copy=True)
            image[image == 255.0] = np.nan
            ca = np.nanmean(np.clip(image[:, :, 0], 0, 1), axis=(1, 2, 3))
            cf = np.nanmean(np.clip(image[:, :, 1], 0, 1), axis=(1, 2, 3))
            cld = np.nanmean(np.clip(image[:, :, 3], 0, 3) / 3.0, axis=(1, 2, 3))
            ca_chunks.append(np.nan_to_num(ca, nan=0.0).astype("float32"))
            cf_chunks.append(np.nan_to_num(cf, nan=0.0).astype("float32"))
            cld_chunks.append(np.nan_to_num(cld, nan=0.0).astype("float32"))
        stats = pd.DataFrame({
            "image_file_key": image_file,
            "image_row": row_ids,
            "cloud_ca_mean": np.concatenate(ca_chunks),
            "cloud_cf_mean": np.concatenate(cf_chunks),
            "cloud_cld_mean": np.concatenate(cld_chunks),
        })
        rows.append(stats)
    return pd.concat(rows, ignore_index=True)


def attach_cloud_stats(frame: pd.DataFrame, stats: pd.DataFrame) -> pd.DataFrame:
    out = frame.merge(stats, on=["image_file_key", "image_row"], how="left")
    for col in ["cloud_ca_mean", "cloud_cf_mean", "cloud_cld_mean"]:
        out[col] = out[col].fillna(0.0).astype("float32")
    return out


def apply_solar_weather_cloud_weight(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    solar_strength = np.sin(np.deg2rad(out["solar_elevation"].astype(float))).clip(0.0, 1.0)
    clear_sky_cf = out.get("target_p90", pd.Series(0.7, index=out.index)).astype(float).clip(0.05, 1.05)

    cloud_severity = (
        0.45 * out["cloud_ca_mean"].astype(float).clip(0, 1)
        + 0.45 * out["cloud_cf_mean"].astype(float).clip(0, 1)
        + 0.10 * out["cloud_cld_mean"].astype(float).clip(0, 1)
    ).clip(0.0, 1.0)
    rain_severity = (np.log1p(out["asos_rn"].fillna(0).astype(float).clip(lower=0)) / np.log1p(10.0)).clip(0.0, 1.0)
    humidity_severity = ((out["asos_hm"].fillna(55).astype(float) - 65.0) / 35.0).clip(0.0, 1.0)
    weather_cloud_severity = (0.70 * cloud_severity + 0.20 * rain_severity + 0.10 * humidity_severity).clip(0.0, 1.0)

    expected_cf = (clear_sky_cf * (0.20 + 0.80 * solar_strength) * (1.0 - 0.70 * weather_cloud_severity)).clip(0.0, 1.05)
    residual = (out["target_capacity_factor"].astype(float).clip(0.0, 1.2) - expected_cf).abs()
    tolerance = (0.10 + 0.08 * (1.0 - solar_strength) + 0.10 * weather_cloud_severity).clip(0.08, 0.28)
    consistency = np.exp(-0.5 * (residual / tolerance) ** 2)

    # 흐림/강수로 낮은 발전량이 설명되는 샘플은 살리고, 맑은 고일사 조건에서 거래량만 비정상적으로 낮은 샘플은 약하게 본다.
    explained_low = (out["target_capacity_factor"].astype(float) < expected_cf * 0.75) & (weather_cloud_severity > 0.45)
    clear_sky_contradiction = (solar_strength > 0.55) & (weather_cloud_severity < 0.25) & (out["target_capacity_factor"].astype(float) < clear_sky_cf * 0.45)

    weight = 0.45 + 1.20 * consistency
    weight = np.where(explained_low, np.maximum(weight, 1.05), weight)
    weight = np.where(clear_sky_contradiction, np.minimum(weight, 0.55), weight)

    out["solar_strength"] = solar_strength.astype("float32")
    out["cloud_severity"] = cloud_severity.astype("float32")
    out["weather_cloud_severity"] = weather_cloud_severity.astype("float32")
    out["expected_cf_weather_adjusted"] = expected_cf.astype("float32")
    out["solar_weather_cloud_residual"] = residual.astype("float32")
    out["solar_weather_cloud_weight"] = np.clip(weight, 0.35, 1.65).astype("float32")
    out["train_weight"] = (out["base_train_weight"] * out["solar_weather_cloud_weight"]).clip(0.05, 5.0).astype("float32")
    return out


cloud_stats = build_cloud_stats(train_df)
for extra_frame in [clean_val_df, real_val_df, real_fair_df]:
    extra_keys = extra_frame[["image_file_key", "image_row"]].drop_duplicates()
    missing = extra_keys.merge(cloud_stats[["image_file_key", "image_row"]], on=["image_file_key", "image_row"], how="left", indicator=True)
    missing = missing.query("_merge == 'left_only'").drop(columns=["_merge"])
    if len(missing):
        cloud_stats = pd.concat([cloud_stats, build_cloud_stats(missing)], ignore_index=True).drop_duplicates(["image_file_key", "image_row"])

train_df = apply_solar_weather_cloud_weight(attach_cloud_stats(train_df, cloud_stats))
clean_val_df = apply_solar_weather_cloud_weight(attach_cloud_stats(clean_val_df, cloud_stats))
real_val_df = apply_solar_weather_cloud_weight(attach_cloud_stats(real_val_df, cloud_stats))
real_fair_df = apply_solar_weather_cloud_weight(attach_cloud_stats(real_fair_df, cloud_stats))

print("\nsolar/weather/cloud weight diagnostics")
print(train_df[["base_train_weight", "solar_weather_cloud_weight", "train_weight", "solar_weather_cloud_residual", "cloud_severity", "weather_cloud_severity"]].describe().T[["min", "mean", "max"]].to_string())
print("\nlowest consistency samples")
print(train_df.sort_values("solar_weather_cloud_weight").head(10)[["target_timestamp_kst", "region", "horizon_hours", "target_capacity_factor", "solar_elevation", "cloud_severity", "weather_cloud_severity", "expected_cf_weather_adjusted", "solar_weather_cloud_weight"]].to_string(index=False))


train: 234694
clean val strong: 48957
real val no_filter: 54989
real fair val: 11880 / 54989
regions: {'대전시': 0, '부산시': 1, '서울시': 2, '울산시': 3, '제주도': 4}
horizons: {1: 0, 2: 1, 3: 2, 4: 3, 5: 4, 6: 5, 7: 6, 8: 7, 9: 8, 10: 9, 11: 10, 12: 11, 13: 12, 14: 13, 15: 14, 16: 15, 17: 16, 18: 17, 19: 18, 20: 19, 21: 20, 22: 21, 23: 22, 24: 23}

feature ranges:
                             min      mean       max
anchor_day_of_year_cos -1.000000 -0.087647  0.999853
anchor_day_of_year_sin -0.999963  0.379799  0.999963
anchor_hour_cos        -1.000000  0.006663  1.000000
anchor_hour_sin        -1.000000 -0.013882  1.000000
asos_hm_scaled          0.070000  0.535792  1.000000
asos_rn_log1p           0.000000  0.021597  3.148453
asos_ta_scaled          0.268571  0.665708  0.965714
cap_scaled              0.043857  0.326276  0.998830
day_of_year_cos        -1.000000 -0.090666  0.999853
day_of_year_sin        -0.999963  0.379002  0.999963
doy_scaled              0.002732  0.345176  0.830601
horizon_ho

In [10]:
class SolarImageDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, num_cols: list[str]):
        self.image_file = frame["image_file_key"].to_numpy()
        self.image_row = frame["image_row"].astype("int64").to_numpy()
        self.num = frame[num_cols].astype("float32").to_numpy()
        self.region = frame["region_id"].astype("int64").to_numpy()
        self.horizon = frame["horizon_id"].astype("int64").to_numpy()
        self.y = frame["target_capacity_factor"].astype("float32").to_numpy()
        self.weight = frame.get("train_weight", pd.Series(1.0, index=frame.index)).astype("float32").to_numpy()

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx: int):
        image = shards[self.image_file[idx]][int(self.image_row[idx])].astype("float32", copy=True)
        image[image == 255.0] = 0.0
        image[:, 0] = image[:, 0].clip(0, 1)      # CA is 0/1 in this image bundle
        image[:, 1] = image[:, 1].clip(0, 1)      # CF is 0/1 in this image bundle
        image[:, 2] = image[:, 2] / 9.0           # CT 1..9
        image[:, 3] = image[:, 3] / 3.0           # CLD 0..3, usually 0..2 here
        image = image.reshape(-1, image.shape[-2], image.shape[-1])

        return (
            torch.from_numpy(image),
            torch.from_numpy(self.num[idx]),
            torch.tensor(self.region[idx], dtype=torch.long),
            torch.tensor(self.horizon[idx], dtype=torch.long),
            torch.tensor(self.y[idx], dtype=torch.float32),
            torch.tensor(self.weight[idx], dtype=torch.float32),
        )


class SatelliteSolarModel(nn.Module):
    def __init__(self, num_dim: int, n_regions: int = 5, n_horizons: int = 24):
        super().__init__()
        self.image = nn.Sequential(
            nn.Conv2d(12, 48, 3, padding=1),
            nn.BatchNorm2d(48),
            nn.SiLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(48, 96, 3, padding=1),
            nn.BatchNorm2d(96),
            nn.SiLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(96, 192, 3, padding=1),
            nn.BatchNorm2d(192),
            nn.SiLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.region = nn.Embedding(n_regions, 8)
        self.horizon = nn.Embedding(n_horizons, 8)
        self.tab = nn.Sequential(
            nn.Linear(num_dim + 16, 128),
            nn.SiLU(),
            nn.Linear(128, 128),
            nn.SiLU(),
        )
        self.short_head = nn.Sequential(
            nn.Linear(192 + 128, 128),
            nn.SiLU(),
            nn.Dropout(0.1),
            nn.Linear(128, 1),
        )
        self.long_head = nn.Sequential(
            nn.Linear(192 + 128, 128),
            nn.SiLU(),
            nn.Dropout(0.15),
            nn.Linear(128, 1),
        )

    def forward(self, image, num, region, horizon):
        image_feat = self.image(image).flatten(1)
        tab_input = torch.cat([num, self.region(region), self.horizon(horizon)], dim=1)
        tab_feat = self.tab(tab_input)
        shared = torch.cat([image_feat, tab_feat], dim=1)
        short_pred = self.short_head(shared).squeeze(1)
        long_pred = self.long_head(shared).squeeze(1)
        return torch.where(horizon >= 6, long_pred, short_pred)


def make_loader(frame: pd.DataFrame, num_cols: list[str], shuffle: bool) -> DataLoader:
    return DataLoader(
        SolarImageDataset(frame, num_cols),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=NUM_WORKERS > 0,
    )


@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader):
    model.eval()
    preds = []
    targets = []
    for image, num, region, horizon, y, weight in loader:
        pred = model(
            image.to(DEVICE, non_blocking=True),
            num.to(DEVICE, non_blocking=True),
            region.to(DEVICE, non_blocking=True),
            horizon.to(DEVICE, non_blocking=True),
        )
        preds.append(pred.detach().float().cpu())
        targets.append(y.detach().float().cpu())

    pred = torch.cat(preds).numpy().clip(0, 1.2)
    y = torch.cat(targets).numpy()
    mae = float(np.mean(np.abs(pred - y)))
    rmse = float(np.sqrt(np.mean((pred - y) ** 2)))
    return mae, rmse, pred, y


In [11]:
def eval_saved_model(name: str, num_cols: list[str]) -> pd.DataFrame:
    out_dir = OUT / name
    ckpt = torch.load(out_dir / "best_model.pt", map_location=DEVICE, weights_only=False)
    model = SatelliteSolarModel(
        num_dim=len(num_cols),
        n_regions=len(region_map),
        n_horizons=len(horizon_map),
    ).to(DEVICE)
    model.load_state_dict(ckpt["model_state"])

    rows = []
    eval_sets = [
        ("clean_strong_val", clean_val_df),
        ("real_no_filter_fair_val", real_fair_df),
        ("real_no_filter_val", real_val_df),
    ]
    for label, frame in eval_sets:
        loader = make_loader(frame, num_cols, shuffle=False)
        mae, rmse, pred, y = evaluate(model, loader)
        rows.append({"model": name, "eval_set": label, "rows": len(frame), "mae": mae, "rmse": rmse})

        predictions = frame[["target_timestamp_kst", "region", "horizon_hours", "hour", "target_capacity_factor"]].copy()
        predictions["pred"] = pred
        predictions["abs_error"] = np.abs(pred - y)
        predictions.to_csv(out_dir / f"pred_{label}.csv", index=False)

    result = pd.DataFrame(rows)
    result.to_csv(out_dir / "eval_summary.csv", index=False)
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result


def train_one(name: str, num_cols: list[str]) -> pd.DataFrame:
    out_dir = OUT / name
    out_dir.mkdir(parents=True, exist_ok=True)
    if not FORCE_RETRAIN and (out_dir / "best_model.pt").exists() and (out_dir / "eval_summary.csv").exists():
        print("\n==========", name, "==========")
        print("using cached model:", out_dir / "best_model.pt")
        return pd.read_csv(out_dir / "eval_summary.csv")

    train_loader = make_loader(train_df, num_cols, shuffle=True)
    clean_loader = make_loader(clean_val_df, num_cols, shuffle=False)

    model = SatelliteSolarModel(
        num_dim=len(num_cols),
        n_regions=len(region_map),
        n_horizons=len(horizon_map),
    ).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    loss_fn = nn.SmoothL1Loss(beta=0.05, reduction="none")

    best_rmse = float("inf")
    bad_epochs = 0
    history = []

    print("\n==========", name, "==========")
    print("num cols:", len(num_cols), num_cols)
    print("params:", round(sum(p.numel() for p in model.parameters()) / 1e6, 3), "M")

    for epoch in range(1, EPOCHS + 1):
        model.train()
        train_loss_sum = 0.0
        train_count = 0

        train_loss_weighted_sum = 0.0
        train_weight_sum = 0.0

        for image, num, region, horizon, y, weight in train_loader:
            image = image.to(DEVICE, non_blocking=True)
            num = num.to(DEVICE, non_blocking=True)
            region = region.to(DEVICE, non_blocking=True)
            horizon = horizon.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            weight = weight.to(DEVICE, non_blocking=True)

            pred = model(image, num, region, horizon)
            loss_per_row = loss_fn(pred, y)
            loss = (loss_per_row * weight).sum() / weight.sum().clamp_min(1.0)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

            train_loss_weighted_sum += float((loss_per_row.detach() * weight).sum().cpu())
            train_weight_sum += float(weight.sum().detach().cpu())
            train_loss_sum += float(loss.detach().cpu()) * len(y)
            train_count += len(y)

        scheduler.step()
        train_mae, train_rmse, _, _ = evaluate(model, train_loader)
        val_mae, val_rmse, _, _ = evaluate(model, clean_loader)
        lr = scheduler.get_last_lr()[0]
        train_loss = train_loss_weighted_sum / max(train_weight_sum, 1.0)
        history.append({"epoch": epoch, "train_loss": train_loss, "train_mae": train_mae, "train_rmse": train_rmse, "val_mae": val_mae, "val_rmse": val_rmse, "lr": lr})
        print(f"{name} epoch {epoch:02d} | train mae {train_mae:.5f} rmse {train_rmse:.5f} | clean val mae {val_mae:.5f} rmse {val_rmse:.5f} | lr {lr:.7f}")

        if val_rmse < best_rmse - MIN_DELTA:
            best_rmse = val_rmse
            bad_epochs = 0
            torch.save(
                {
                    "model_state": model.state_dict(),
                    "num_cols": num_cols,
                    "region_map": region_map,
                    "horizon_map": horizon_map,
                    "model_name": "satellite_wind_safe_multihorizon_24h_v10_solar_weather_cloud_weighted",
                    "image_normalization": "binary",
                    "head_policy": "short_head_for_1_6h_long_head_for_7_24h",
                    "loss": "SmoothL1Loss(beta=0.05) weighted by sample_weight_strong * horizon_balance_weight * solar_weather_cloud_weight",
                    "bundle": str(DATA),
                },
                out_dir / "best_model.pt",
            )
        else:
            bad_epochs += 1
            if bad_epochs >= PATIENCE:
                print("early stop:", epoch)
                break

    pd.DataFrame(history).to_csv(out_dir / "history.csv", index=False)
    del model, optimizer, scheduler, train_loader, clean_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    result = eval_saved_model(name, num_cols)
    print("\neval summary:")
    print(result.to_string(index=False))
    return result


In [12]:
results = []
for name, cols in EXPERIMENTS:
    results.append(train_one(name, cols))

summary = pd.concat(results, ignore_index=True).sort_values(["eval_set", "rmse"])
summary_path = OUT / "model_compare_summary.csv"
summary.to_csv(summary_path, index=False)

print("\n========== FINAL MODEL COMPARISON ==========")
print(summary.to_string(index=False))
print("saved summary:", summary_path)



========== satellite_wind_safe_multihorizon_24h_v10_solar_weather_cloud_weighted ==========
num cols: 25 ['cap_scaled', 'solar_elev_scaled', 'is_daylight', 'hour_scaled', 'doy_scaled', 'month_scaled', 'hour_of_day_sin', 'hour_of_day_cos', 'day_of_year_sin', 'day_of_year_cos', 'wind_u_scaled', 'wind_v_scaled', 'wind_speed_scaled', 'wind_dir_sin', 'wind_dir_cos', 'asos_ta_scaled', 'asos_hm_scaled', 'asos_rn_log1p', 'horizon_hours_scaled', 'horizon_hours_sin', 'horizon_hours_cos', 'anchor_hour_sin', 'anchor_hour_cos', 'anchor_day_of_year_sin', 'anchor_day_of_year_cos']
params: 0.318 M
satellite_wind_safe_multihorizon_24h_v10_solar_weather_cloud_weighted epoch 01 | train mae 0.19721 rmse 0.23807 | clean val mae 0.21100 rmse 0.25348 | lr 0.0001998
satellite_wind_safe_multihorizon_24h_v10_solar_weather_cloud_weighted epoch 02 | train mae 0.15797 rmse 0.19265 | clean val mae 0.18228 rmse 0.21608 | lr 0.0001993
satellite_wind_safe_multihorizon_24h_v10_solar_weather_cloud_weighted epoch 03 | t

In [13]:
summary = pd.read_csv(OUT / "model_compare_summary.csv")

print("OUT exists:", OUT.exists(), OUT)
print("summary exists:", (OUT / "model_compare_summary.csv").exists())
print()
print("===== MODEL SUMMARY =====")
print(summary.sort_values(["eval_set", "rmse"]).to_string(index=False))

pivot_rmse = summary.pivot(index="eval_set", columns="model", values="rmse")
pivot_mae = summary.pivot(index="eval_set", columns="model", values="mae")
print()
print("===== RMSE PIVOT =====")
print(pivot_rmse.to_string())
print()
print("===== MAE PIVOT =====")
print(pivot_mae.to_string())

def group_metrics(frame: pd.DataFrame, group_col: str) -> pd.DataFrame:
    work = frame.copy()
    work["_sq_error"] = (work["pred"] - work["target_capacity_factor"]) ** 2
    result = (
        work.groupby(group_col, as_index=False)
        .agg(
            rows=("abs_error", "size"),
            mae=("abs_error", "mean"),
            mse=("_sq_error", "mean"),
            target_mean=("target_capacity_factor", "mean"),
            pred_mean=("pred", "mean"),
        )
    )
    result["rmse"] = np.sqrt(result["mse"])
    result = result.drop(columns=["mse"])
    ordered_cols = [group_col, "rows", "mae", "rmse", "target_mean", "pred_mean"]
    return result[ordered_cols].sort_values("rmse", ascending=False)


for model in [name for name, _ in EXPERIMENTS]:
    print()
    print()
    print("==========", model, "==========")
    model_dir = OUT / model

    hist_path = model_dir / "history.csv"
    if hist_path.exists():
        hist = pd.read_csv(hist_path)
        best = hist.loc[hist["val_rmse"].idxmin()]
        print()
        print("===== BEST EPOCH =====")
        print(best.to_string())
        print()
        print("===== HISTORY TAIL =====")
        print(hist.tail(10).to_string(index=False))

    for eval_set in ["clean_strong_val", "real_no_filter_fair_val", "real_no_filter_val"]:
        p = model_dir / f"pred_{eval_set}.csv"
        if not p.exists():
            print("missing:", p)
            continue

        df = pd.read_csv(p)
        rmse = np.sqrt(((df["pred"] - df["target_capacity_factor"]) ** 2).mean())
        print()
        print("=====", eval_set, "=====")
        print("rows:", len(df))
        print("mae:", df["abs_error"].mean())
        print("rmse:", rmse)
        print()
        print("by horizon")
        print(group_metrics(df, "horizon_hours").to_string(index=False))
        print()
        print("by region")
        print(group_metrics(df, "region").to_string(index=False))
        print()
        print("worst 15")
        print(df.sort_values("abs_error", ascending=False).head(15).to_string(index=False))


OUT exists: True /home/j-k14s305/s305-work/runs/satellite_wind_safe_multihorizon_24h_v10_solar_weather_cloud_weighted
summary exists: True

===== MODEL SUMMARY =====
                                                                model                eval_set  rows      mae     rmse
satellite_wind_safe_multihorizon_24h_v10_solar_weather_cloud_weighted        clean_strong_val 48957 0.079356 0.100202
satellite_wind_safe_multihorizon_24h_v10_solar_weather_cloud_weighted real_no_filter_fair_val 11880 0.109242 0.140547
satellite_wind_safe_multihorizon_24h_v10_solar_weather_cloud_weighted      real_no_filter_val 54989 0.094894 0.124597

===== RMSE PIVOT =====
model                    satellite_wind_safe_multihorizon_24h_v10_solar_weather_cloud_weighted
eval_set                                                                                      
clean_strong_val                                                                      0.100202
real_no_filter_fair_val                              